# Proposal Revision Classification (RQ1)

Replicates the API-documentation evolution methodology on the proposals
(`all_proposals.db`):

* **Step 1 — Identify revisions.** For each proposal we diff *consecutive*
  revisions, ordered by `revision_index` with a per-proposal direction chosen so
  `created_at` is monotonically increasing (some `created_at` are date-only and
  tie-prone, and some projects — Rust — store the index in reverse). Content
  changes come from diffing `ProposalRevision.content`; metadata changes
  (title / author / status) come from the structured tables.
* **Step 2 — Classify by heuristics.** Each change is labeled with a *part*
  (code / prose / metadata) and an *operation* (added / deleted / modified);
  modified prose is sub-typed (typo / formatting / rephrase / substantive).
* **Step 3 — Refine.** The sampling cell at the bottom prints random classified
  changes for manual inspection so the rules in `revision_analysis/classify.py`
  can be tuned.

This notebook answers:
* **RQ1 — Which parts of proposals are frequently revised?** (incl. in-progress vs terminal)
* **Extra — How frequently are proposals changed?** (incl. in-progress vs terminal)

In [ ]:
import sys
from pathlib import Path

# make the revision_analysis package importable from the notebooks/ dir
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from revision_analysis import pipeline, db

sns.set_theme(style="whitegrid")
PLOTS = Path("plots")
PLOTS.mkdir(exist_ok=True)
CACHE = Path("cache")
PROJECT_ID = None  # None = all 10 projects; set an int to focus on one

In [ ]:
# Run the pipeline (or load cached parquet).
# Results are cached so re-running the analysis is instant.
def load_or_run(force=False):
    files = {n: CACHE / f"{n}.parquet" for n in
             ("content_changes", "metadata_changes", "revision_pairs")}
    if not force and all(f.exists() for f in files.values()):
        return {n: pd.read_parquet(f) for n, f in files.items()}
    return pipeline.run(project_id=PROJECT_ID, cache_dir=str(CACHE))


res = load_or_run(force=False)
content = res["content_changes"]
meta = res["metadata_changes"]
pairs = res["revision_pairs"]

with db.connect() as con:
    PROJ = db.project_names(con)
content["project"] = content["project_id"].map(PROJ)
meta["project"] = meta["project_id"].map(PROJ)
pairs["project"] = pairs["project_id"].map(PROJ)
len(content), len(meta), len(pairs)

## RQ1 — Which parts of proposals are frequently revised?

In [ ]:
# Combine content + metadata changes into one tidy frame of classified changes.
all_changes = pd.concat([
    content[["project", "category", "op", "block_kind"]],
    meta[["project", "category", "op"]].assign(block_kind="metadata"),
], ignore_index=True)

cat_counts = all_changes["category"].value_counts()
print(cat_counts)

ax = cat_counts.plot.barh(figsize=(8, 6))
ax.set_title("Revision categories across all proposals")
ax.set_xlabel("number of changes")
plt.tight_layout()
plt.savefig(PLOTS / "rq1-categories.svg")
plt.show()

In [ ]:
# Part-level share (code / prose / metadata) overall and per project.
# Fold inline metadata_header into the structured "metadata" part.
def part(row):
    return "metadata" if "metadata" in row["block_kind"] else row["block_kind"]


all_changes["part"] = all_changes.apply(part, axis=1)
overall = all_changes["part"].value_counts(normalize=True).mul(100).round(1)
print("Overall % by part:\n", overall, "\n")

per_proj = (all_changes.groupby("project")["part"].value_counts(normalize=True)
            .mul(100).unstack().fillna(0).round(1))
ax = per_proj.plot.barh(stacked=True, figsize=(9, 6))
ax.set_title("Share of revision parts per project")
ax.set_xlabel("% of changes")
ax.legend(title="part", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.savefig(PLOTS / "rq1-parts-per-project.svg")
plt.show()
per_proj

In [ ]:
# Prose modification sub-types (typo / formatting / rephrase / substantive).
prose_mods = content[(content["block_kind"] == "prose") &
                     (content["op"] == "modified")]
sub = prose_mods["prose_subtype"].value_counts()
print(sub)
ax = sub.plot.bar(figsize=(7, 4), rot=0)
ax.set_title("Prose modification sub-types")
ax.set_ylabel("count")
plt.tight_layout()
plt.savefig(PLOTS / "rq1-prose-subtypes.svg")
plt.show()

In [ ]:
# Sub-category breakdown of revisions as a single 100% composition: the three
# major parts (prose / code / metadata) sum to 100% together, each bar's length
# being that part's share of all changes, segmented by sub-category. To keep large
# projects from dominating, per-category counts are first divided by each project's
# proposal count and averaged across projects (equal project weight) before the
# percentages are taken.
allc = pd.concat([content[["project_id", "category"]],
                  meta[["project_id", "category"]]], ignore_index=True)

n_props = pairs.groupby("project_id")["proposal_id"].nunique()
grid = pd.MultiIndex.from_product([n_props.index, sorted(allc["category"].unique())],
                                  names=["project_id", "category"])
counts = allc.groupby(["project_id", "category"]).size().reindex(grid, fill_value=0)
per_proposal = counts.div(n_props, level="project_id")  # rate within each project
mean_rate = per_proposal.groupby("category").mean()  # equal-weight mean across projects

# stable sub-category order within each major part
order = ["prose_added", "prose_deleted", "prose_modified", "prose_substantive",
         "prose_rephrase", "prose_typo", "prose_formatting",
         "code_added", "code_deleted", "code_modified",
         "metadata_header_added", "metadata_header_deleted", "metadata_header_modified",
         "metadata_title", "metadata_author", "metadata_status"]
mr = mean_rate.reset_index(name="rate")
mr["major"] = mr["category"].str.split("_").str[0]
pivot = (mr.pivot_table(index="major", columns="category", values="rate", fill_value=0)
         .reindex(columns=[c for c in order if c in set(mr["category"])]))
# percentage of ALL changes, so the three major bars sum to 100% together
pct = pivot.div(pivot.values.sum()).mul(100).reindex(["metadata", "code", "prose"])

LABELS = {
    "prose_added": "Added",
    "prose_deleted": "Deleted",
    "prose_modified": "Modified (cross-kind)",
    "prose_substantive": "Substantive",
    "prose_rephrase": "Rephrase",
    "prose_typo": "Typo",
    "prose_formatting": "Formatting",
    "code_added": "Added",
    "code_deleted": "Deleted",
    "code_modified": "Modified",
    "metadata_header_added": "Header added",
    "metadata_header_deleted": "Header deleted",
    "metadata_header_modified": "Header modified",
    "metadata_title": "Title",
    "metadata_author": "Author",
    "metadata_status": "Status",
}

ax = (pct.rename(columns=LABELS, index=str.capitalize)
      .plot.barh(stacked=True, figsize=(11, 4), colormap="tab20", width=0.7))
ax.set_xlim(0, 100)
ax.set_xlabel("% of all changes (project-normalized)")
ax.set_ylabel("")
# ax.set_title("Revision sub-categories by part (share of all changes)")
ax.legend(title="Sub-category", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(PLOTS / "rq1-subcategories-normalized.svg")
plt.show()
print("major-part totals (%):", pct.sum(axis=1).round(1).to_dict())

latex_rows = []
for major in ["metadata", "code", "prose"]:
    if major not in pct.index:
        continue
    subs = [(LABELS.get(c, c), pct.loc[major, c])
            for c in pct.columns if pct.loc[major, c] > 0]
    if not subs:
        continue
    major_total = pct.loc[major].sum()
    latex_rows.append(
        f"\\multirow{{{len(subs)}}}{{*}}{{\\textbf{{{major.capitalize()}}}}} "
        f"& \\multirow{{{len(subs)}}}{{*}}{{{major_total:.2f}\\%}} "
        f"& {subs[0][0]} & {subs[0][1]:.2f}\\% \\\\"
    )
    for label, value in subs[1:]:
        latex_rows.append(f" & & {label} & {value:.2f}\\% \\\\")
    latex_rows.append("\\midrule")
if latex_rows and latex_rows[-1] == "\\midrule":
    latex_rows.pop()

latex = "\n".join([
    "\\begin{table}[h]",
    "\\centering",
    "\\caption{Revision sub-categories as percentages of all changes (project-normalized).}",
    "\\label{tab:rq1-subcategories}",
    "\\begin{tabular}{lllr}",
    "\\toprule",
    "Part & Part \\% & Sub-category & \\% of all changes \\\\",
    "\\midrule",
    *latex_rows,
    "\\bottomrule",
    "\\end{tabular}",
    "\\end{table}",
])
print("\n" + latex)

pct.round(2)

## Terminal vs in-progress: do revision category proportions differ?

Compare the distribution of change categories between revisions made while a
proposal was still **in progress** and those made once it had reached a
**terminal** state. Crucially, this split is per *revision*, not per proposal:
each change is assigned the proposal's status *at the time the revision was
made*. A proposal can move review → accepted → review, so we treat it as
in-progress until the trailing run of statuses is entirely terminal
(accepted / rejected / withdrawn / superseded) — only revisions at or after that
point count as terminal. Proposals that never permanently settle contribute all
their revisions to the in-progress group.

In [ ]:
# Classify each change as terminal vs in-progress by the proposal's status at
# the time of the revision. A revision is terminal only if it was made once the proposal
# had *permanently* reached a terminal status (accepted/rejected/withdrawn/superseded).
# Because a proposal can bounce review -> accepted -> review, `db.terminal_cutoffs`
# returns the start of the trailing all-terminal run (NaT if it never permanently settles);
# a revision at time r is terminal iff r >= cutoff, else in-progress.
with db.connect() as con:
    cutoffs = db.terminal_cutoffs(con)
cutoff_map = cutoffs.set_index(["project_id", "proposal_id"])["terminal_cutoff"].to_dict()

# Time a change happened: the newer revision's timestamp for content diffs, and
# the status/title/author change timestamp for metadata.
content_ts = content.assign(
    change_time=pd.to_datetime(content["to_created"], errors="coerce", utc=True))
meta_ts = meta.assign(
    change_time=pd.to_datetime(meta["created_at"], errors="coerce", utc=True))

all_changes_with_status = pd.concat([
    content_ts[["project_id", "proposal_id", "project", "category", "op", "block_kind", "change_time"]],
    meta_ts[["project_id", "proposal_id", "project", "category", "op", "change_time"]].assign(block_kind="metadata"),
], ignore_index=True)
all_changes_with_status["part"] = all_changes_with_status.apply(
    lambda r: "metadata" if "metadata" in r["block_kind"] else r["block_kind"], axis=1)


def status_at_revision(r):
    cutoff = cutoff_map.get((r["project_id"], r["proposal_id"]))
    if cutoff is None or pd.isna(cutoff):  # never permanently terminal
        return "in_progress"
    if pd.isna(r["change_time"]):
        return "unknown"
    return "terminal" if r["change_time"] >= cutoff else "in_progress"


all_changes_with_status["status_group"] = all_changes_with_status.apply(status_at_revision, axis=1)

# Filter to the two main groups
two_groups = all_changes_with_status[
    all_changes_with_status["status_group"].isin(["terminal", "in_progress"])
]

# Part-level comparison
part_by_status = pd.crosstab(
    two_groups["status_group"], two_groups["part"], normalize="index"
).mul(100).round(1)
print("Part-level share (%) by status group:")
print(part_by_status)

# Category-level comparison
cat_by_status = pd.crosstab(
    two_groups["status_group"], two_groups["category"], normalize="index"
).mul(100).round(2)

CAT_LABELS = {
    "prose_added": "Prose: added",
    "prose_deleted": "Prose: deleted",
    "prose_modified": "Modified (cross-kind)",
    "prose_substantive": "Prose: substantive",
    "prose_rephrase": "Prose: rephrase",
    "prose_typo": "Prose: typo",
    "prose_formatting": "Prose: formatting",
    "code_added": "Code: added",
    "code_deleted": "Code: deleted",
    "code_modified": "Code: modified",
    "metadata_header_added": "Metadata header: added",
    "metadata_header_deleted": "Metadata header: deleted",
    "metadata_header_modified": "Metadata header: modified",
    "metadata_title": "Metadata: title",
    "metadata_author": "Metadata: author",
    "metadata_status": "Metadata: status",
}
STATUS_LABELS = {"in_progress": "In progress", "terminal": "Terminal"}

CAT_ORDER = [
    "metadata_status",
    "prose_formatting",
    "prose_typo",
    "prose_rephrase",
    "code_modified",
    "metadata_header_modified",
    "metadata_author",
    "prose_modified",
    "prose_substantive",
    "code_added",
    "metadata_title",
    "prose_added",
    "prose_deleted",
    "code_deleted",
    "metadata_header_added",
    "metadata_header_deleted",
]

plot_df = (cat_by_status.T
           .reindex([c for c in CAT_ORDER if c in cat_by_status.columns])
           .reindex(columns=["terminal", "in_progress"])
           .rename(index=CAT_LABELS, columns=STATUS_LABELS))

fig, ax = plt.subplots(figsize=(12, 5))
plot_df.iloc[::-1].plot.barh(ax=ax)
ax.set_xlabel("% of changes within status group")
ax.set_ylabel("")
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1], title="Status group")
# ax.set_title("Revision category proportions: terminal vs in-progress proposals")
plt.tight_layout()
plt.savefig(PLOTS / "rq1-terminal-vs-inprogress-categories.svg")
plt.show()

print(
    cat_by_status.T.assign(
        pct_change=lambda d: ((d["terminal"] - d["in_progress"]) / d["in_progress"] * 100).round(2)
    ).sort_values("pct_change", ascending=False)
    .to_latex(
        float_format="%.2f",
        caption="Revision category proportions: terminal vs in-progress proposals",
        label="results:rq1-terminal-vs-inprogress",
        position="h",
    )
)

cat_by_status.T.assign(
    pct_change=lambda d: ((d["terminal"] - d["in_progress"]) / d["in_progress"] * 100).round(2)
).sort_values("pct_change", ascending=False)

## Extra — How frequently are proposals changed?

In [ ]:
# Distribution of revisions per proposal, overall and per project.
print(pairs["n_revisions"].describe())
ax = sns.boxplot(data=pairs, x="project", y="n_revisions")
ax.set_yscale("log")
# ax.set_title("Revisions per proposal by project")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(PLOTS / "extra-revisions-per-project.svg")
plt.show()

In [ ]:
# In-progress (draft/review) vs terminal (accepted/rejected/withdrawn/superseded).
summary = (pairs.groupby("status_group")[["n_revisions", "lifespan_days",
                                          "revisions_per_month"]].median())
print("Medians by status group:\n", summary, "\n")
print(pairs.groupby(["status_group", "latest_status"]).size())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(data=pairs, x="status_group", y="n_revisions", ax=axes[0])
axes[0].set_yscale("log")
axes[0].set_title("# revisions")
sns.boxplot(data=pairs, x="status_group", y="revisions_per_month", ax=axes[1])
axes[1].set_yscale("log")
axes[1].set_title("revisions / month of life")
plt.tight_layout()
plt.savefig(PLOTS / "extra-status-group.svg")
plt.show()

## Refinement loop

Run the cell below to print random classified changes and eyeball whether the
assigned category is right. When you spot systematic mistakes, tune the
thresholds / rules in `revision_analysis/classify.py` and re-run with
`load_or_run(force=True)`.

In [ ]:
from revision_analysis.diff import diff_revisions
from revision_analysis.segment import format_for_project


# Inspect a specific proposal: re-diff its consecutive revisions and print the
# old/new text next to the assigned category. This is the manual-refinement view
# (full text is not cached in parquet, so we recompute on demand). Revisions are
# fetched in the corrected chronological order (db.ordered_revisions).
def inspect(project_id, proposal_id, max_changes=20):
    fmt = format_for_project(project_id)  # fixed format for single-format projects
    with db.connect() as con:
        revs = db.ordered_revisions(con, project_id, proposal_id)
    shown = 0
    for a, b in zip(revs, revs[1:]):
        print("=" * 80)
        for ch in diff_revisions(a.content, b.content, fmt):
            print("-" * 80)
            print(f"{ch.category}  (op={ch.op}, kind={ch.block_kind},"
                  f" subtype={ch.prose_subtype})")
            if ch.old:
                print("  OLD:", ch.old.replace("\n", " "))
            if ch.new:
                print("  NEW:", ch.new.replace("\n", " "))
            shown += 1
            if shown >= max_changes:
                return


# Pick a random multi-revision proposal to inspect.
cand = content.drop_duplicates(["project_id", "proposal_id"]).sample(1).iloc[0]
print(f"Inspecting [{cand['project']}] proposal {cand['proposal_id']}\n")
inspect(int(cand["project_id"]), cand["proposal_id"])